# CONNECT TO running server as client

In [1]:
# server has to run first (builddrun powershell)

In [2]:
import socketio
import time
import threading
import numpy as np 

# Setup Socket.IO client
sio = socketio.Client()
uid = 'jupyter-client'

latest_data = None
message_log = []

# 1️⃣ Define listeners BEFORE connect
@sio.on('ex', namespace='/main')
def on_ex(data):
    global latest_data, message_log
    #print(f"📩 Server returned on 'ex': {data}")
    latest_data = data
    message_log.append({'event': 'ex', 'data': data})

@sio.event(namespace='/main')
def connect():
    print("✅ Connected to /main")
    sio.emit('join', {'usr': uid}, namespace='/main')

# 2️⃣ Connect
sio.connect('http://127.0.0.1:5000', namespaces=['/main'])

# 3️⃣ Start background thread AFTER connect
def wait_forever():
    while True:
        time.sleep(1)

thread = threading.Thread(target=wait_forever)
thread.daemon = True
thread.start()

#print("📡 Listening for messages in the background...")


✅ Connected to /main


### sending from jupyter-client to others (e.g. webclient)

In [3]:
# basic example 
# sio.emit('ex', {
#     'usr': 'jupyter-client',
#     'msg': 'Node: 20', 'id': None, 'val': '20', 'fn': 'node'
# }, namespace='/main')


### change project 

In [4]:
# get all projects in backend 

import GlobalData as GD

# print as pretty table with index from  0 - x
allprojects = []
for i, proj in enumerate(GD.listProjects()):
    allprojects.append((i,proj))
allprojects

[(0, 'CDK5'),
 (1, 'CircLadderGraph-xsmall'),
 (2, 'CLGraph_TEST'),
 (3, 'diffusion'),
 (4, 'Exposurome'),
 (5, 'GenExpression_01'),
 (6, 'GenExpression_02'),
 (7, 'imunet_250130-X0-inter'),
 (8, 'imunet_250130-X1-inter'),
 (9, 'Interactive_Project_T01'),
 (10, 'JSON_autocore'),
 (11, 'JSON_barbellgraph'),
 (12, 'JSON_Zachary'),
 (13, 'Microplastics'),
 (14, 'Powergrid_Europe'),
 (15, 'Realtime-project'),
 (16, 'Sphere_Torus'),
 (17, 'Sphere_Torus_Morph'),
 (18, 'Teapot'),
 (19, 'TheMandelbulb_edges')]

In [5]:
# choose a project id from list above to change project 
sel_index = 12

sel_id = allprojects[sel_index][0]
sel_name = allprojects[sel_index][1]

sio.emit('ex', {
    'usr': 'jupyter-client',
    'fn': 'dropdown',
    'val': sel_id,
    'msg' : sel_name,
    'id': 'projDD',
}, namespace='/main')

### CREATE NEW PROJECT

In [6]:
import networkx as nx 
import pandas as pd

# G = nx.circular_ladder_graph(20)
# G.graph['projectname'] = 'Realtime-project'
# print("number of G nodes = ", len(G.nodes))   
# print("number of G edges = ", len(G.edges))

# TEAPOT GRAPH
G = nx.read_edgelist("temp-files/teapot-raw/teapot40_links.csv", delimiter=',', nodetype=int)
print("Number of nodes: ", len(G.nodes()))
print("Number of Links: ", len(G.edges()))

G.graph['projectname'] = "TEAPOT-realtime"
G.graph['info'] = "A toy graph for testing purposes. Number of nodes: "+str(len(G.nodes()))+", Links: "+ str(len(G.edges()))+"."

nodepos = pd.read_csv("temp-files/teapot-raw/teapot40_nodes.csv", delimiter=',',header =None)
pos = dict(zip(G.nodes(), zip(nodepos[0], nodepos[1], nodepos[2])))
nx.set_node_attributes(G, pos, 'pos')

Number of nodes:  51361
Number of Links:  102560


In [7]:
# make project in backend
import nx2json as nx2j 

nx2j.create_project(G)

✅ Connected to /main
Successfully created the directory static/projects/TEAPOT-realtime 
PROGRESS: loaded graph JSON...
PROGRESS: stored graph data...
PROGRESS: stored layouts...
PROGRESS: stored node info...
PROGRESS: made node position textures...
PROGRESS: made textures for node colors...
PROGRESS: made textures for links...
PROGRESS: no linkcolors detected for  layoutname_0
PROGRESS: writing json files for project and nodes...
Project created successfully.


### GET ALL PROJECTS AND SELECT THE NEW ONE CREATED 

In [8]:
# see if new project is in projectlist 

allprojects_updated = []
for i, proj in enumerate(GD.listProjects()):
    allprojects_updated.append((i,proj))
allprojects_updated

[(0, 'CDK5'),
 (1, 'CircLadderGraph-xsmall'),
 (2, 'CLGraph_TEST'),
 (3, 'diffusion'),
 (4, 'Exposurome'),
 (5, 'GenExpression_01'),
 (6, 'GenExpression_02'),
 (7, 'imunet_250130-X0-inter'),
 (8, 'imunet_250130-X1-inter'),
 (9, 'Interactive_Project_T01'),
 (10, 'JSON_autocore'),
 (11, 'JSON_barbellgraph'),
 (12, 'JSON_Zachary'),
 (13, 'Microplastics'),
 (14, 'Powergrid_Europe'),
 (15, 'Realtime-project'),
 (16, 'Sphere_Torus'),
 (17, 'Sphere_Torus_Morph'),
 (18, 'Teapot'),
 (19, 'TEAPOT-realtime'),
 (20, 'TheMandelbulb_edges')]

In [9]:
# choose a project id from list above to change project 
sel_index = 19

sel_id = allprojects_updated[sel_index][0]
sel_name = allprojects_updated[sel_index][1]

sio.emit('ex', {
    'usr': 'jupyter-client',
    'fn': 'dropdown',
    'val': sel_id,
    'msg' : sel_name,
    'id': 'projDD',
}, namespace='/main')

✅ Connected to /main


### MODIFY PROJECT REAL TIME
- node colors 
- link colors
- node positions

In [10]:
# change linkcolors 

visible_links = list(G.edges())#[::10]
print(len(visible_links))
print(visible_links[:10])

linkcolors = [(255,0,255,100)]

new_texture_name = 'teapot_showALLlinks'

102560
[(0, 1), (0, 3), (0, 6519), (1, 2), (1, 82), (2, 3), (2, 4), (2, 83), (3, 5), (3, 6520)]


✅ Connected to /main


In [11]:
from PIL import Image

hight = 64 * (int((len(list(G.edges()))) / 32768) + 1)
path = 'static/projects/' + sel_name 
texc = [(0,0,0,10)] * 512 * hight  
new_imgc = Image.new('RGBA', (512, hight))

link_rgba = [(l,c) for l in visible_links for c in linkcolors]
    
edge_to_index = {tuple(edge): i for i, edge in enumerate(G.edges())}
for l, c in link_rgba:
    if tuple(l) in edge_to_index:
        i = edge_to_index[tuple(l)]
        texc[i]  = (int(c[0]),int(c[1]),int(c[2]),int(c[3]))

new_imgc.putdata(texc)
pathRGB = path + '/linksRGB/' + new_texture_name +'.png' 
new_imgc.save(pathRGB, "PNG")


In [12]:
# SAVE TO PERMAMENT FILE STRUCTURE

# add to pfile channel 
import json
pfile = path + '/pfile.json'
with open(pfile) as f:
    data = json.load(f)

    # if not exists add
    if new_texture_name not in data['linksRGB']:
         data['linksRGB'].append(new_texture_name)
    
    # also add to other channels
    if new_texture_name not in data['layouts']:
        data['layouts'].append(new_texture_name)
    if new_texture_name not in data['layoutsRGB']:
        data['layoutsRGB'].append(new_texture_name)

with open(pfile, 'w') as f:
    json.dump(data, f, indent=4)


# add textures to other channels for file number consistency
import shutil
import os

# read all files in a folders
current_layouts = os.listdir(path + '/layouts')
current_layoutsrgb = os.listdir(path + '/layoutsRGB')
current_layoutsl = os.listdir(path + '/layoutsl')
print("Current layouts: ", current_layouts)

# copy one of the files via index and save as new name = same as texture generated 
shutil.copy(path + "/layouts/" + current_layouts[0], path + "/layouts/" + new_texture_name + ".bmp")
shutil.copy(path + "/layoutsRGB/" + current_layoutsrgb[0], path + "/layoutsRGB/" + new_texture_name + ".png")
shutil.copy(path + "/layoutsl/" + current_layoutsl[0], path + "/layoutsl/" + new_texture_name + "l.bmp")

Current layouts:  ['layoutname_0.bmp']


'static/projects/TEAPOT-realtime/layoutsl/teapot_showALLlinksl.bmp'

In [13]:
# UPDATE TEXTURE AND SEND TO FRONTEND 

sio.emit('ex', {
    'usr': 'jupyter-client',
    'fn': "updateTempTex",
    'id': 'linksRGBDD',
    'textures':[{"channel": "linkRGB", 
                  'path': "static/projects/"+ sel_name  + "/linksRGB/" + new_texture_name + ".png"}]
}, namespace='/main')

In [14]:
# RELOAD CURRENT PROJECT 

sio.emit('ex', {
    'usr': 'jupyter-client',
    'fn': 'dropdown',
    'val': sel_id,
    'msg' : sel_name,
    'id': 'projDD',
}, namespace='/main')


### REAL-TIME LAYOUTS

In [15]:
# MODIFY NODE POSITION 

new_layout_name = "C_temp"


# FUNCTION FOR NODE XYZ texture
hight = 128 * (int((len(G.nodes())) / 16384) + 1)
size = 128 * hight 
path = 'static/projects/' + sel_name 
    
texh = [(0,0,0)] * size
texl = [(0,0,0)] * size

# add layout here !!!!!!!!!
# the below part is dummy data for testing
nodes_pos = nx.get_node_attributes(G, 'pos') 
new_node_pos = {k: (v[0], v[2], v[1]) for k, v in nodes_pos.items()}

for i in range(len(new_node_pos)):

    x = int(float(new_node_pos[i][0])*65280)
    y = int(float(new_node_pos[i][1])*65280)
    z = int(float(new_node_pos[i][2])*65280)

    xh = int(x / 255)
    yh = int(y / 255)
    zh = int(z / 255)

    xl = x % 255
    yl = y % 255
    zl = z % 255

    pixelh = (xh,yh,zh)
    pixell = (xl,yl,zl)

    texh[i] = pixelh
    texl[i] = pixell

new_imgh = Image.new('RGB', (128, hight))
new_imgl = Image.new('RGB', (128, hight))
new_imgh.putdata(texh)
new_imgl.putdata(texl)

pathXYZ = path + '/layouts/' +  new_layout_name + '.bmp'
pathXYZl = path + '/layoutsl/' +  new_layout_name  + 'l.bmp' 
new_imgh.save(pathXYZ)
new_imgl.save(pathXYZl)


In [16]:
# CHANGE NODE COLORS 

new_nodecolors = [[68,51,255,80]] * len(G.nodes())

# FUNCTION FOR NODE COLORS TEXTURE
hight = 128 * (int((len(G.nodes())) / 16384) + 1)
size = 128 * hight 
path = 'static/projects/' + sel_name
tex = [(0,0,0,10)] * size

for i in range(len(new_nodecolors)): 
    tex[i] = (int(new_nodecolors[i][0]), int(new_nodecolors[i][1]),int(new_nodecolors[i][2]),int(new_nodecolors[i][3]))

new_img = Image.new('RGBA', (128, hight))
new_img.putdata(tex)
pathRGB = path + '/layoutsRGB/' +  new_layout_name + '.png'
new_img.save(pathRGB , "PNG")


In [17]:
# ADD ANOTHER LINKLIST TEXTURE 

visible_links = list(G.edges())[::50]

hight = 64 * (int((len(list(G.edges()))) / 32768) + 1)
path = 'static/projects/' + sel_name 
texc = [(0,0,0,10)] * 512 * hight  
new_imgc = Image.new('RGBA', (512, hight))

link_rgba = [(l,c) for l in visible_links for c in linkcolors]
    
edge_to_index = {tuple(edge): i for i, edge in enumerate(G.edges())}
for l, c in link_rgba:
    if tuple(l) in edge_to_index:
        i = edge_to_index[tuple(l)]
        texc[i]  = (int(c[0]),int(c[1]),int(c[2]),int(c[3]))

new_imgc.putdata(texc)
pathRGB = path + '/linksRGB/' + new_layout_name +'.png' 
new_imgc.save(pathRGB, "PNG")

✅ Connected to /main


In [18]:
# SAVE TO PERMAMENT FILE STRUCTURE

# add to pfile channel 
import json
pfile = path + '/pfile.json'
with open(pfile) as f:
    data = json.load(f)

    # if not exists add
    if new_layout_name not in data['linksRGB']:
         data['linksRGB'].append(new_layout_name)
    if new_layout_name not in data['layouts']:
        data['layouts'].append(new_layout_name)
    if new_layout_name not in data['layoutsRGB']:
        data['layoutsRGB'].append(new_layout_name)

with open(pfile, 'w') as f:
    json.dump(data, f, indent=4)


✅ Connected to /main


In [19]:
# RELOAD CURRENT PROJECT 

sio.emit('ex', {
    'usr': 'jupyter-client',
    'fn': 'dropdown',
    'val': sel_id,
    'msg' : sel_name,
    'id': 'projDD',
}, namespace='/main')


# TO DO 

In [20]:




# Related to Content
# continue workflow + FOCUS VR-Jupyter client COMMUNICATION!!!



# for example 
# start from scratch in project - maybe random layout 
    # 1. in jupyter client - create project - by default creates random layout
    # 2. in VR client - see the project 

    # 3. in jupyter client - make global layout for a better vis representation
  
    # 4. in VR client look at patterns - select a node

# 5. in jupyter client - get all neighbors of selected node - then relayout with subgraph in center and sphere around 

# 6. in VR client - see the new layout

# 7. ??????????


In [55]:
from dataXplorer import SessionManager, TextureGenerator, ProjectFileManager, VisualizerSyncer

# Setup
session = SessionManager(sel_id, sel_name, sio)
session.load_graph(G)
tex_gen = TextureGenerator(session)
file_mgr = ProjectFileManager(session)
syncer = VisualizerSyncer(session)

# choose layout name 
layout_name = "01-newsubset"

# modify link texture
tex_gen.generate_link_texture(list(G.edges())[::10], (0,0,220, 200), layout_name)

# modify node color texture
some_node_list = list(G.nodes())[::100]
color_map = {n: (0,0,220, 200) for n in some_node_list}
tex_gen.generate_node_color_texture(color_map, layout_name)

# modify node position texture
# dummy relayouting - replace with actual layout algorithms
nodepos = nx.get_node_attributes(G, 'pos')
posG_new = dict(zip(G.nodes(),nx.rescale_layout(np.array(list(nodepos.values())), scale=0.9)))
tex_gen.generate_node_position_texture(posG_new, layout_name)

generated = tex_gen.get_generated_types()
file_mgr.update_pfile(layout_name)

file_mgr.sync_layout_files(layout_name, generated_types=generated)

syncer.emit_texture_update(layout_name)
syncer.reload_project()

✅ Connected to /main


✅ Connected to /main


# useful snippets

### receive messages from server or web/VR client

In [33]:
# SEND DATA - select a node via jupyter client
# sio.emit('ex', {
#     'usr': 'jupyter-client',
#     'fn': 'node',
#     'val': '2534', #1398',
#     'msg' : '', #'CDK5',
#     'id': None,
# }, namespace='/main')

In [34]:
# SELECT SOMETHING on Web/VR client and check if received here on jupyter client : 
#print("📦 Latest data received:", latest_data)

✅ Connected to /main
✅ Connected to /main
